# Урок 8. KMeans: находим группы без ответов

До сих пор у нас был **ответ** (`y`): выжил, возраст. Модель училась на примерах с ответами. Это обучение **с учителем**.
Теперь ответа нет. Мы просим модель сама найти группы похожих людей. Это обучение **без учителя**, а группы называются **кластерами**.

**Как работает KMeans** (K = сколько групп ты просишь):
1. Бросаем K точек-«центров» в случайные места.
2. Каждого человека относим к ближайшему центру.
3. Двигаем каждый центр в середину его людей.
4. Повторяем 2–3, пока центры не перестанут двигаться.

«Ближе» значит **меньше расстояние** между людьми по признакам. Поэтому масштаб признаков очень важен (часть 1).

## 0. Подготовка

`Survived` в модель **не даём**. Он нужен только в конце, чтобы посмотреть: совпали ли кластеры с выжившими.

Признаки: `Pclass`, `Age`, `Fare`, `FamilySize` (числа) и `Sex` (one-hot). `Title` не берём, чтобы не усложнять.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../data/titanic.csv")
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

num_cols = ["Pclass", "Age", "Fare", "FamilySize"]
cat_cols = ["Sex"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(), cat_cols),
])
X = preprocess.fit_transform(df)     # вся таблица в числах, без Survived
print(X.shape)

(891, 6)


## 1. Зачем масштабирование

Покажем на примере. Сначала **без** масштабирования: просто 4 числовых признака как есть.
`Fare` бывает до 512, а `Pclass` только 1–3. Расстояние почти целиком зависит от `Fare`.

In [2]:
raw = df[num_cols].fillna(df["Age"].median())
km_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(raw)

print("людей в кластерах:", pd.Series(km_raw.labels_).value_counts().sort_index().to_dict())
raw.assign(кластер=km_raw.labels_).groupby("кластер").mean().round(1)

людей в кластерах: {0: 729, 1: 20, 2: 142}


,Pclass,Age,Fare,FamilySize
кластер,,,,
0,2.6,28.3,15.5,1.8
1,1.0,30.8,279.3,2.8
2,1.2,34.7,83.4,2.5


**Вывод:** кластеры 729, 142 и 20 человек. Они делятся в основном по цене билета (средняя 15, 83 и 279 фунтов). Возраст различается слабо (28–35 лет). Класс различается только потому, что он связан с ценой. Модель почти «не заметила» возраст и размер семьи: их числа маленькие рядом с `Fare`.
`StandardScaler` приводит все признаки к одной шкале. Для KMeans это **обязательно**.

## 2. Сколько кластеров? Метрики

KMeans не знает, сколько групп на самом деле. K выбираем мы. Две метрики помогают:

- **inertia** — сумма квадратов расстояний от людей до своих центров. Чем **меньше**, тем «теснее» кластеры. Но она **всегда падает** при росте K (при K = 891 она ноль).
- **silhouette** — число от −1 до 1. Чем ближе к 1, тем чётче кластеры отделены друг от друга. Здесь **больше = лучше**.

In [3]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    rows.append({"K": k,
                 "inertia": km.inertia_,
                 "silhouette": silhouette_score(X, km.labels_)})
pd.DataFrame(rows).round(3)

,K,inertia,silhouette
0,2,2836.535,0.374
1,3,2210.279,0.392
2,4,1813.226,0.376
3,5,1565.259,0.344
4,6,1338.860,0.360
5,7,1206.194,0.372
6,8,1087.815,0.373


**Вывод:**
- `inertia` падает всегда (2837 → 1088). Из неё нельзя выбрать K: «локтя» (резкого излома) нет.
- `silhouette` лучший при K=3: **0.392**. При K=2 и K=4 почти так же (0.374 и 0.376). Разница меньше 0.02: считай это шумом.
- Все значения 0.34–0.39: это **слабая** структура. Чётких групп на Титанике нет. Люди образуют скорее непрерывное облако. Берём K=3 как разумный вариант.

## 3. Кто в кластерах? Профиль

Кластер — просто номер. Что он значит, мы узнаём, посмотрев на средние по признакам. Смотри внимательно: `людей` это **количество**, остальные колонки средние или **доли**.

In [4]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
df["cluster"] = km.labels_

profile = df.groupby("cluster").agg(
    людей=("Pclass", "size"),
    класс=("Pclass", "mean"),
    возраст_медиана=("Age", "median"),
    цена_медиана=("Fare", "median"),
    семья=("FamilySize", "mean"),
    доля_женщин=("Sex", lambda s: (s == "female").mean()),
)
profile.round(2)

,людей,класс,возраст_медиана,цена_медиана,семья,доля_женщин
cluster,,,,,,
0,553,2.75,27.0,8.05,1.34,0.28
1,237,1.10,39.0,55.44,1.78,0.44
2,101,2.71,8.0,27.90,5.32,0.54


**Вывод (читаем строки):**
- **Кластер 0** (553 человека): 3-й класс (2.75), билет дешёвый (8), семья 1.3, в основном мужчины (28% женщин). Одиночки из 3-го класса.
- **Кластер 1** (237): 1-й класс (1.1), старше (39), билет 55. Пассажиры 1-го класса.
- **Кластер 2** (101): 3-й класс, но возраст **8 лет** и семья **5.3** человека. Большие семьи с детьми.

Названия мы придумали сами. KMeans их не даёт.

## 4. А выжили ли они? Сравнение с `Survived`

Мы **не** давали `Survived` модели. Посмотрим, совпали ли группы с ним.

In [5]:
df.groupby("cluster")["Survived"].agg(["mean", "size"]).round(2).rename(
    columns={"mean": "доля_выживших", "size": "людей"})

,доля_выживших,людей
cluster,,
0,0.28,553
1,0.61,237
2,0.42,101


**Вывод:** доля выживших: кластер 0 — 0.28, кластер 1 — 0.61, кластер 2 — 0.42. Различия есть, и они логичны (1-й класс выживал чаще).
Но это **не предсказание**. KMeans не знал, кто выжил. Он группировал по сходству признаков, а выжившие совпали с группами «заодно».
Для предсказания выживания у нас есть классификация (уроки 1–6).

## 5. Ловушка: `n_init` и случайный старт

Начальные центры берутся **случайно**. Плохой старт даёт плохие кластеры. `n_init=10` значит: запустить 10 раз с разными стартами и взять лучший результат (с меньшей inertia).
Сравним `n_init=1` с разными `random_state`.

In [6]:
for rs in range(5):
    km1 = KMeans(n_clusters=4, n_init=1, random_state=rs).fit(X)
    print(f"random_state={rs}: inertia = {km1.inertia_:.1f}")

random_state=0: inertia = 1954.6
random_state=1: inertia = 1813.3
random_state=2: inertia = 1812.7
random_state=3: inertia = 1819.8
random_state=4: inertia = 1813.3


**Вывод:** при `random_state=0` inertia 1954.6, при остальных около 1813. Один старт попал в **плохой** вариант: кластеры хуже на 8%.
Поэтому оставляем `n_init=10` (в новых версиях sklearn по умолчанию `"auto"`, но лучше писать число явно).

## 6. Шпаргалка

```python
X = preprocess.fit_transform(df)                 # масштабирование обязательно, y не нужен
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
km.labels_                                       # номер кластера каждого человека
km.inertia_                                      # теснота кластеров (меньше = теснее)
silhouette_score(X, km.labels_)                  # чёткость (больше = чётче)
km.predict(X_new)                                # кластер для нового человека
```

KMeans **не умеет**: находить «форму» кластеров (ищет шары), сам выбирать K, работать с категориями без one-hot.

---
# 🏠 Домашка

## Задание 1. Без пола

Повтори таблицу из части 2 (K от 2 до 8, inertia и silhouette), но **без** признака `Sex`: только 4 числовых признака.
1. Какой K лучший по silhouette?
2. Изменилось ли значение silhouette заметно по сравнению с таблицей из урока?
3. **Вывод словами.**

## Задание 2. Читаем таблицу

Сделай профиль (как в части 3) для K=4. Для каждого из 4 кластеров напиши одну строку: **кто эти люди** (с числами из таблицы).
Для одного кластера обязательно проверь: ты пишешь «людей» (количество) или «доля»?

## Задание 3. Вопрос без кода

Друг посмотрел часть 4 и сказал: «Кластер 1 выжил на 61%. Значит, KMeans научился предсказывать выживание!»
Что ты ответишь? Назови **две** причины, почему это неверно.

**Ответ:**

In [7]:
numeric_preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
])

X2 = numeric_preprocess.fit_transform(df)

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X2)
    rows.append({"K": k,
                 "inertia": km.inertia_,
                 "silhouette": silhouette_score(X2, km.labels_)})
pd.DataFrame(rows).round(3)

,K,inertia,silhouette
0,2,2433.817,0.424
1,3,1817.831,0.451
2,4,1426.802,0.435
3,5,1191.349,0.411
4,6,1036.807,0.408
5,7,906.967,0.388
6,8,806.852,0.398


### Ответ на задание 1

1. Лучший K по silhouette: **K = 3** (0.451). K = 4 близко (0.435): разница 0.016, это почти шум.
2. Без `Sex` silhouette стал заметно выше: **0.451 против 0.392**. Разница 0.06 больше, чем разница между соседними K, значит, это не шум.
3. **Вывод:** без пола кластеры чётче. Вероятно, пол (0/1) добавляет «прыжок» в расстоянии, который не совпадает с группами по классу и возрасту. Но 0.451 всё равно слабая или средняя структура, не чёткие группы.

**Важно:** inertia из двух таблиц сравнивать нельзя. Без `Sex` 4 колонки, с `Sex` — 6. Чем больше колонок, тем больше сумма расстояний, поэтому «inertia стала меньше» ничего не значит.

In [8]:
# задание 2: профиль для K=4 (признаки как в части 0, с Sex)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
df["cluster"] = km4.labels_

profile4 = df.groupby("cluster").agg(
    людей=("Pclass", "size"),
    класс=("Pclass", "mean"),
    возраст_медиана=("Age", "median"),
    цена_медиана=("Fare", "median"),
    семья=("FamilySize", "mean"),
    доля_женщин=("Sex", lambda s: (s == "female").mean()),
)
profile4.round(2)

,людей,класс,возраст_медиана,цена_медиана,семья,доля_женщин
cluster,,,,,,
0,233,1.30,42.0,32.32,1.58,0.39
1,513,2.81,26.0,8.05,1.33,0.27
2,47,1.00,31.0,153.46,2.60,0.66
3,98,2.73,8.0,28.45,5.37,0.53


### Ответ на задание 2

- **Кластер 0** (233 человека): в основном 1-й класс (1.30), медианный возраст 42, билет 32.3, семья 1.6 человека. Пожилые пассажиры 1-го класса.
- **Кластер 1** (513): 3-й класс (2.81), возраст 26, билет 8.05, семья 1.3, женщин 27%. Одиночки из 3-го класса.
- **Кластер 2** (47): 1-й класс (1.00), возраст 31, билет 153.5, семья 2.6, женщин 66%. Самые богатые, маленькая группа.
- **Кластер 3** (98): 3-й класс (2.73), возраст 8 (медиана), билет 28.5, семья 5.4 человека. Дети из больших семей.

Как читать таблицу: `людей` — это **количество**, `доля_женщин` — **доля**, остальное — средние и медианы. Номер кластера ничего не значит и может поменяться при другом запуске.

### Ответ на задание 3

Друг неправ по двум причинам:
1. KMeans **не знал**, кто выжил: `Survived` не давали модели. Она группировала людей по сходству признаков (класс, возраст, цена, семья, пол).
2. 61% — это **доля выживших в кластере** на тех же данных. Это описание, а не точность предсказания. Для проверки нужны новые люди (тест) и сравнение с базовой линией (в целом выжили 38%).

Для предсказания выживания у нас есть классификация (уроки 1–6).